In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 15 consolidation — Turn disparity error into a physical uncertainty calculation

Read `sessions/15_session.md` first. This is an executed reference, not an assessment of you.

**Evidence:** Real stereo-reference disparities with generated local noise; no independent calibration-noise measurement.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT/'data/raw/iris.csv').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Start Jupyter in the extracted listening_to_shape_lab folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))


## Step 1

Select finite reference disparities and compute the local derivative with explicit units.

Before running: write your expected result and name one way this check could fail.

In [3]:
from shape_lab.stereo import reference_sample, depth_from_disparity
_, _, reference, calibration = reference_sample()
samples = reference[::80, ::80].ravel()
samples = samples[np.isfinite(samples)][:25]
f = calibration['focal_px']; B = calibration['baseline_m']; offset = calibration['doffs_px']
sigma_px = .2
expected_sigma_m = f*B/(samples+offset)**2 * sigma_px
assert len(samples)>5
print('first reference depths:', depth_from_disparity(samples, f, B, offset)[:5])

first reference depths: [4.63603096 4.57042944 4.3599105  3.72509585 3.6324089 ]


## Step 2

Generate only the declared perturbation, not replacement reference truth.

Before running: write your expected result and name one way this check could fail.

In [4]:
rng = np.random.default_rng(1504)
perturbed = samples[None,:] + rng.normal(0, sigma_px, size=(5000,len(samples)))
depths = depth_from_disparity(perturbed, f, B, offset)
measured_sigma_m = np.std(depths, axis=0, ddof=1)
ratio = measured_sigma_m/expected_sigma_m
print('empirical / first-order standard deviation:', ratio)
assert np.all(np.abs(ratio-1)<.06)

empirical / first-order standard deviation: [1.00041967 0.99792935 0.99850997 1.00949673 0.989364   1.00526803
 1.00193992 1.00327321 1.00745169 1.00066796 1.00405096 1.01098463
 0.99101904 1.00635764 0.99934757 0.99812103 0.99375082 1.00032621
 0.99698366 1.00912336 0.9983801  0.99425539 1.00231646 1.0061225
 1.00382266]


## Step 3

Store what is fixed and what is random so the numerical agreement cannot be mistaken for complete sensor uncertainty.

Before running: write your expected result and name one way this check could fail.

In [5]:
result = {'points': len(samples), 'perturbation_trials': len(perturbed),
          'disparity_sigma_px': sigma_px, 'median_empirical_to_linear_sigma_ratio': float(np.median(ratio)),
          'calibration_fixed': True, 'noise_model': 'synthetic independent Gaussian disparity noise',
          'actual_matcher_uncertainty_established': False}

## Keep execution evidence separate from learning evidence

The following saves this reference calculation. Write your own explanation in `my_work/`; no learner score is assigned.

In [6]:
result['stage'] = 15
result['execution_role'] = 'reference_consolidation'
result['data_role'] = 'Real stereo-reference disparities with generated local noise; no independent calibration-noise measurement.'
output = ROOT/'reports/consolidation_v4/15.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Saved reference result:', output.relative_to(ROOT))


Saved reference result: reports/consolidation_v4/15.json
